# Online Retail Data Processing

This notebook cleans and explores the UCI Online Retail dataset.
The dataset contains transactions from a UK-based online store between 2010-2011.

**Dataset columns:**
- `InvoiceNo` – transaction ID
- `StockCode` – product code
- `Description` – product name
- `Quantity` – units purchased
- `InvoiceDate` – date and time of transaction
- `UnitPrice` – price per unit (GBP)
- `CustomerID` – unique customer identifier
- `Country` – customer country

## 1. Load the data

In [1]:
import pandas as pd
from download_data import download_and_extract

In [2]:
data_url = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"
path = download_and_extract(data_url)

Data set already exists: data/Online Retail.xlsx 
Skipping downloading.


In [4]:
df = pd.read_excel(path)
print(df.shape)
df.head()

(541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## 2. Initial exploration

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


In [6]:
df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


In [7]:
# Check missing values
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

**Observations:**
- `CustomerID` has ~135k missing values — these are guest/anonymous transactions
- `Description` has ~1.4k missing values
- Some `Quantity` values are negative (returns/cancellations)
- Some `UnitPrice` values are 0

## 3. Data Cleaning

In [8]:
# drop rows with missing CustomerID, for customer-level analysis
df = df.dropna(subset=['CustomerID'])
print(f'Rows after dropping missing CustomerID: {len(df)}')

Rows after dropping missing CustomerID: 406829


In [7]:
# remove cancelled orders, InvoiceNo starts with 'C'
cancelled = df['InvoiceNo'].astype(str).str.startswith('C')
print(f'Cancelled orders: {cancelled.sum()}')
df = df[~cancelled]
print(f'Rows after removing cancellations: {len(df)}')

Cancelled orders: 8905
Rows after removing cancellations: 397924


In [9]:
# remove rows where Quantity or UnitPrice which is less than 0
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
print(f'Rows after removing invalid quantity/price: {len(df)}')

Rows after removing invalid quantity/price: 397884


In [10]:
# fix data types for processing
df['CustomerID'] = df['CustomerID'].astype(int).astype(str)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

df.dtypes

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID                str
Country                   str
dtype: object

## 4. Feature Engineering

In [11]:
# Create a revenue column
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# Extract date parts — useful for time-based analysis
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()

df[['InvoiceDate', 'Revenue', 'Year', 'Month', 'DayOfWeek']].head()

,InvoiceDate,Revenue,Year,Month,DayOfWeek
0,2010-12-01 08:26:00,15.30,2010,12,Wednesday
1,2010-12-01 08:26:00,20.34,2010,12,Wednesday
2,2010-12-01 08:26:00,22.00,2010,12,Wednesday
3,2010-12-01 08:26:00,20.34,2010,12,Wednesday
4,2010-12-01 08:26:00,20.34,2010,12,Wednesday


## 5. Basic Analysis

In [12]:
# total revenue
print(f"Total Revenue: £{df['Revenue'].sum():,.2f}")

Total Revenue: £8,911,407.90


In [13]:
# calculate monthly revenue
monthly_revenue = df.groupby(['Year', 'Month'])['Revenue'].sum().reset_index()
monthly_revenue.columns = ['Year', 'Month', 'TotalRevenue']
monthly_revenue

,Year,Month,TotalRevenue
0,2010,12,572713.890
1,2011,1,569445.040
2,2011,2,447137.350
3,2011,3,595500.760
4,2011,4,469200.361
5,2011,5,678594.560
6,2011,6,661213.690
7,2011,7,600091.011
8,2011,8,645343.900
9,2011,9,952838.382


In [13]:
# top 10 countries by revenue exclude UK since it dominates
top_countries = (
    df[df['Country'] != 'United Kingdom']
    .groupby('Country')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
top_countries

Country
Netherlands    285446.34
EIRE           265545.90
Germany        228867.14
France         209024.05
Australia      138521.31
Spain           61577.11
Switzerland     56443.95
Belgium         41196.34
Sweden          38378.33
Japan           37416.37
Name: Revenue, dtype: float64

In [14]:
# Top 10 best-selling products by quantity
top_products = (
    df.groupby('Description')['Quantity']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
top_products

Description
PAPER CRAFT , LITTLE BIRDIE           80995
MEDIUM CERAMIC TOP STORAGE JAR        77916
WORLD WAR 2 GLIDERS ASSTD DESIGNS     54415
JUMBO BAG RED RETROSPOT               46181
WHITE HANGING HEART T-LIGHT HOLDER    36725
ASSORTED COLOUR BIRD ORNAMENT         35362
PACK OF 72 RETROSPOT CAKE CASES       33693
POPCORN HOLDER                        30931
RABBIT NIGHT LIGHT                    27202
MINI PAINT SET VINTAGE                26076
Name: Quantity, dtype: int64

In [15]:
# Number of unique customers
print(f"Unique customers: {df['CustomerID'].nunique()}")

# Average order value per customer
avg_order = df.groupby('InvoiceNo')['Revenue'].sum().mean()
print(f"Average order value: £{avg_order:.2f}")

Unique customers: 4338
Average order value: £480.87


## 6. Save cleaned data

In [16]:
df.to_csv('online_retail_cleaned.csv', index=False)
print('Saved cleaned data to online_retail_cleaned.csv')

Saved cleaned data to online_retail_cleaned.csv
